In [5]:
import numpy as np
import pandas as pd
import glob
import os
import traceback

# ======================================================
# CONFIG
# ======================================================

CATALOG_FILE = "2my_galaxies_catalog-Copy1.cat"
LINES_DIR = "LINES_FITS"
SPECTRA_DIR = "SPECTRA"

H0 = 70.0                      # km/s/Mpc
OmegaM = 0.25
OmegaL = 0.75
c = 299792.458                 # km/s
Mpc_to_cm = 3.08568e24
LN10 = np.log(10)

REST = {
    "Hb": 4861.333,
    "O3": 5006.843,
    "Ha": 6562.819,
    "N2": 6583.460,
    "SIIa": 6716.440,
    "SIIb": 6730.810
}

# ======================================================
# COSMOLOGY FUNCTION
# ======================================================

def luminosity_distance(z):

    z_grid = np.linspace(0, z, 1000)
    Ez = np.sqrt(OmegaM*(1+z_grid)**3 + OmegaL)
    integral = np.trapezoid(1/Ez, z_grid)

    D_H = c / H0                      # Mpc
    D_C = D_H * integral              # comoving distance
    D_L = (1+z) * D_C                 # luminosity distance in Mpc

    return D_L, D_L * Mpc_to_cm


# ======================================================
# LOAD CATALOG
# ======================================================

catalog = pd.read_csv(CATALOG_FILE, sep=r"\s+", comment="#")
catalog = catalog.set_index("iden")

# ======================================================
# FAST LINE MATCH
# ======================================================

def get_line_fast(centers, target):
    return np.abs(centers - target).argmin()


# ======================================================
# PROCESS ONE GALAXY
# ======================================================

def process_fit_file(fit_path):

    filename = os.path.basename(fit_path)

    try:

        base = filename.replace("fit_", "").replace(".csv", "")
        parts = base.split("_")

        iden = int(parts[0])
        radius = parts[1]
        xcoord = parts[2]
        ycoord = parts[3]
        smooth = "_smooth" if "smooth" in filename else ""

        if iden not in catalog.index:
            raise ValueError("Galaxy not in catalog")

        z_sys = catalog.loc[iden, "z"]
        RA = catalog.loc[iden, "RA"]
        DEC = catalog.loc[iden, "DEC"]
        zconf = catalog.loc[iden, "zconf"]

        # logM if present
        logM = catalog.loc[iden, "logM"] if "logM" in catalog.columns else np.nan

        # -----------------------------------------------
        # Luminosity Distance
        # -----------------------------------------------
        DL_Mpc, DL_cm = luminosity_distance(z_sys)

        # -----------------------------------------------
        # Load fit
        # -----------------------------------------------
        df = pd.read_csv(fit_path)

        centers = df["Center"].values
        flux = df["Flux"].values
        dflux = df["d_Flux"].values

        exp = {k: v*(1+z_sys) for k,v in REST.items()}
        idx = {k: get_line_fast(centers, exp[k]) for k in REST}

        lines = {}
        for k in REST:
            lines[k] = {
                "center": centers[idx[k]],
                "flux": flux[idx[k]],
                "dflux": dflux[idx[k]]
            }

        # -----------------------------------------------
        # Quality checks (basic S/N)
        # -----------------------------------------------

        SN = {k: lines[k]["flux"]/lines[k]["dflux"] 
              if lines[k]["dflux"] > 0 else 0 for k in REST}

        flag_Hb_SN = SN["Hb"] < 3
        flag_Ha_SN = SN["Ha"] < 3
        flag_N2_SN = SN["N2"] < 3
        flag_O3_SN = SN["O3"] < 3
        flag_SIIa_SN = SN["SIIa"] < 3
        flag_SIIb_SN = SN["SIIb"] < 3

        flag_negative_flux = any(lines[k]["flux"] <= 0 for k in REST)
        
        # -----------------------------------------------
        # Redshifts per line
        # -----------------------------------------------
        z_lines = {
            k: lines[k]["center"]/REST[k] - 1
            for k in REST
        }

        # -----------------------------------------------
        # Rest wavelengths inferred
        # -----------------------------------------------
        rest_obs = {
            k: lines[k]["center"]/(1+z_sys)
            for k in REST
        }

        # -----------------------------------------------
        # Line ratios
        # -----------------------------------------------
        R1 = lines["O3"]["flux"]/lines["Hb"]["flux"]
        R2 = lines["N2"]["flux"]/lines["Ha"]["flux"]
        SII_sum = lines["SIIa"]["flux"] + lines["SIIb"]["flux"]
        R3 = SII_sum/lines["Ha"]["flux"]

        R1_err = R1*np.sqrt(
            (lines["O3"]["dflux"]/lines["O3"]["flux"])**2 +
            (lines["Hb"]["dflux"]/lines["Hb"]["flux"])**2
        )

        R2_err = R2*np.sqrt(
            (lines["N2"]["dflux"]/lines["N2"]["flux"])**2 +
            (lines["Ha"]["dflux"]/lines["Ha"]["flux"])**2
        )

        SII_sum_err = np.sqrt(
            lines["SIIa"]["dflux"]**2 +
            lines["SIIb"]["dflux"]**2
        )

        R3_err = R3*np.sqrt(
            (SII_sum_err/SII_sum)**2 +
            (lines["Ha"]["dflux"]/lines["Ha"]["flux"])**2
        )

        logR1 = np.log10(R1)
        logR2 = np.log10(R2)
        logR3 = np.log10(R3)

        logR1_err = R1_err/(R1*LN10)
        logR2_err = R2_err/(R2*LN10)
        logR3_err = R3_err/(R3*LN10)

        O3N2 = logR1 - logR2
        O3N2_err = np.sqrt(logR1_err**2 + logR2_err**2)

        OH = 8.73 - 0.32*O3N2
        OH_err = 0.32*O3N2_err

        # -----------------------------------------------
        # Spectrum for EW
        # -----------------------------------------------
        spec_name = f"id{iden}_{xcoord}_{ycoord}_circ_{radius}{smooth}"
        spec_path = os.path.join(SPECTRA_DIR, spec_name)

        spec = np.loadtxt(spec_path)
        wave = spec[:,0]
        flux_spec = spec[:,1]

        Hb_center = lines["Hb"]["center"]

        blue = (wave > Hb_center-90) & (wave < Hb_center-10)
        red  = (wave > Hb_center+10) & (wave < Hb_center+90)

        cont_blue = np.median(flux_spec[blue])
        cont_red = np.median(flux_spec[red])
        cont = 0.5*(cont_blue+cont_red)

        err_blue = np.std(flux_spec[blue])
        err_red = np.std(flux_spec[red])
        cont_err = 0.5*np.sqrt(err_blue**2 + err_red**2)

        EW_obs = lines["Hb"]["flux"]/cont
        sigma_EW_obs = EW_obs*np.sqrt(
            (lines["Hb"]["dflux"]/lines["Hb"]["flux"])**2 +
            (cont_err/cont)**2
        )

        EW_rest = EW_obs/(1+z_sys)
        sigma_EW_rest = sigma_EW_obs/(1+z_sys)

        # -----------------------------------------------
        # SFR
        # -----------------------------------------------
        FHa = lines["Ha"]["flux"]*1e-20
        FHb = lines["Hb"]["flux"]*1e-20
        dFHa = lines["Ha"]["dflux"]*1e-20
        dFHb = lines["Hb"]["dflux"]*1e-20

        FHb_corr = FHb + 3*FHb/EW_obs
        dFHb_corr = np.sqrt(
            ((1+3/EW_obs)*dFHb)**2 +
            ((-3*FHb)/EW_obs**2 * sigma_EW_obs)**2
        )

        Av = 6.547*np.log10(FHa/(2.86*FHb_corr))
        d_Av = 6.547/LN10*np.sqrt(
            (dFHa/FHa)**2 +
            (dFHb_corr/FHb_corr)**2
        )

        FHa_der = FHa*10**(0.31*Av)
        dFHa_der = FHa_der*np.sqrt(
            (dFHa/FHa)**2 +
            (0.31*LN10*d_Av)**2
        )

        LHa = 4*np.pi*FHa_der*DL_cm**2
        dLHa = LHa*(dFHa_der/FHa_der)

        SFR = 7.9e-42*LHa
        dSFR = SFR*(dFHa_der/FHa_der)

        logSFR = np.log10(SFR)
        # ---- Asymmetric log errors ----
        SFR_low = SFR - dSFR
        SFR_up  = SFR + dSFR

        if SFR_low <= 0:
            SFR_low = np.nan

        logSFR_low = np.log10(SFR_low)
        logSFR_up  = np.log10(SFR_up)

        d_logSFR_low = logSFR - logSFR_low
        d_logSFR_up  = logSFR_up - logSFR
        

        # -----------------------------------------------
        # Additional quality flags
        # -----------------------------------------------

        #flag_balmer = (FHa / (2.86 * lines["Hb"]["flux"]*1e-20)) < 0.5
        flag_large_errorR1 = any(x > 5 for x in [logR1_err])
        flag_large_errorR2 = any(x > 5 for x in [logR2_err])
        flag_large_errorR3 = any(x > 5 for x in [logR3_err])
        
        quality_flag = any([
            flag_Hb_SN,
            flag_Ha_SN,
            flag_N2_SN,
            flag_O3_SN,
            flag_negative_flux,
            flag_large_errorR1,
            flag_large_errorR2,
            flag_large_errorR3
        ])

        # -----------------------------------------------
        # SAVE EVERYTHING
        # -----------------------------------------------
        result = pd.DataFrame([{
            "iden": iden,
            "RA": RA,
            "DEC": DEC,
            "z": z_sys,
            "zconf": zconf,
            "logM": logM,
            
            "DL_Mpc": DL_Mpc,
            "DL_cm": DL_cm,
                # --- Emission line measurements ---
            **{
                f"{k}_rest": rest_obs[k]
                for k in rest_obs
            },

            **{
                 f"{k}_center": centers[idx[k]]
                for k in lines
            },

            **{
                f"{k}_flux": flux[idx[k]]
                for k in lines
            },

             **{
                 f"{k}_dflux": dflux[idx[k]]
                for k in lines
            },
            
            "R1": R1,
            "R1_err": R1_err,
            "R2": R2,
            "R2_err": R2_err,
            "R3": R3,
            "R3_err": R3_err,
            
            "logR1": logR1,
            "logR1_err": logR1_err,
            "logR2": logR2,
            "logR2_err": logR2_err,
            "logR3": logR3,
            "logR3_err": logR3_err,
            
            "O3N2": O3N2,
            "O3N2_err": O3N2_err,
            "12+log(O/H)": OH,
            "12+log(O/H)_err": OH_err,
            
            "EW_Hb_obs": EW_obs,
            "EW_Hb_obs_err": sigma_EW_obs,
            "EW_Hb_rest": EW_rest,
            "EW_Hb_rest_err": sigma_EW_rest,
            
            "SFR": SFR,
            "d_SFR": dSFR,
            "log10_SFR": logSFR,
            "d_log10_SFR_up": d_logSFR_up,
            "d_log10_SFR_low": d_logSFR_low,
        
            "flag_Hb_SN": flag_Hb_SN,
            "flag_Ha_SN": flag_Ha_SN,
            "flag_N2_SN": flag_N2_SN,
            "flag_O3_SN": flag_O3_SN,
            "flag_SIIa_SN": flag_SIIa_SN,
            "flag_SIIb_SN": flag_SIIb_SN,
            #"flag_balmer": flag_balmer,
            "flag_negative_flux": flag_negative_flux,
            "flag_large_errorR1": flag_large_errorR1,
            "flag_large_errorR2": flag_large_errorR2,
            "flag_large_errorR3": flag_large_errorR3,
            "quality_flag": quality_flag
        }])    

        output_name = f"new_Z_fit_{base}.csv"
        result.to_csv(os.path.join(LINES_DIR, output_name),
                      index=False, float_format="%.3f")

        print(f"✔ {filename}")

    except Exception as e:
        print(f"❌ Error in {filename}: {e}")


# ======================================================
# MAIN
# ======================================================

def main():
    fit_files = glob.glob(os.path.join(LINES_DIR, "fit_*.csv"))
    for f in fit_files:
        process_fit_file(f)

    print("\nPipeline complete.")


if __name__ == "__main__":
    main()


✔ fit_151_r4_197_147.csv
✔ fit_1103_r5_197_285.csv
✔ fit_1081_r5_335_276.csv
✔ fit_202_r8_272_138.csv
❌ Error in fit_212_r4_283_273.csv: SPECTRA/id212_283_273_circ_r4 not found.
✔ fit_202_r9_271_139.csv
✔ fit_553_r3_147_264.csv
✔ fit_202_r4_273_137.csv
✔ fit_1233_r3_72_279_smooth.csv
✔ fit_812_r10_206_308.csv
✔ fit_407_r6_187_141.csv
✔ fit_1483_r5_123_313.csv
✔ fit_590_r6_293_183.csv
✔ fit_1103_r3_198_284_smooth.csv
✔ fit_680_r4_278_220.csv
✔ fit_212_r11_285_274.csv
✔ fit_151_r7_195_149.csv
✔ fit_812_r2_206_308.csv
✔ fit_1081_r2_335_276.csv
✔ fit_212_r7_283_272.csv
✔ fit_285_r2_233_167.csv
✔ fit_553_r6_148_264.csv
✔ fit_285_r6_234_165_smooth.csv
✔ fit_459_r2_152_157.csv
✔ fit_1233_r4_72_279_smooth.csv
✔ fit_1086_r4_263_273.csv
✔ fit_407_r2_188_141.csv
✔ fit_680_r2_278_220.csv
✔ fit_101_r2_211_42_smooth.csv
✔ fit_459_r5_152_157.csv
✔ fit_553_r5_148_264.csv
✔ fit_590_r2_294_183.csv

Pipeline complete.


In [3]:
import numpy as np
import pandas as pd
import glob
import os
import traceback

# ======================================================
# CONFIG
# ======================================================

CATALOG_FILE = "2my_galaxies_catalog-Copy1.cat"
LINES_DIR = "LINES_FITS"
SPECTRA_DIR = "SPECTRA"

H0 = 70.0                      # km/s/Mpc
OmegaM = 0.25
OmegaL = 0.75
c = 299792.458                 # km/s
Mpc_to_cm = 3.08568e24
LN10 = np.log(10)

REST = {
    "Hb": 4861.333,
    "O3": 5006.843,
    "Ha": 6562.819,
    "N2": 6583.460,
    "SIIa": 6716.440,
    "SIIb": 6730.810
}

# ======================================================
# COSMOLOGY FUNCTION
# ======================================================

def luminosity_distance(z):

    z_grid = np.linspace(0, z, 1000)
    Ez = np.sqrt(OmegaM*(1+z_grid)**3 + OmegaL)
    integral = np.trapezoid(1/Ez, z_grid)

    D_H = c / H0                      # Mpc
    D_C = D_H * integral              # comoving distance
    D_L = (1+z) * D_C                 # luminosity distance in Mpc

    return D_L, D_L * Mpc_to_cm


# ======================================================
# LOAD CATALOG
# ======================================================

catalog = pd.read_csv(CATALOG_FILE, sep=r"\s+", comment="#")
catalog = catalog.set_index("iden")

# ======================================================
# FAST LINE MATCH
# ======================================================

def get_line_fast(centers, target):
    return np.abs(centers - target).argmin()


# ======================================================
# PROCESS ONE GALAXY
# ======================================================

def process_fit_file(fit_path):

    filename = os.path.basename(fit_path)

    try:

        base = filename.replace("fit_", "").replace(".csv", "")
        parts = base.split("_")

        iden = int(parts[0])
        radius = parts[1]
        xcoord = parts[2]
        ycoord = parts[3]
        smooth = "_smooth" if "smooth" in filename else ""

        if iden not in catalog.index:
            raise ValueError("Galaxy not in catalog")

        z_sys = catalog.loc[iden, "z"]
        RA = catalog.loc[iden, "RA"]
        DEC = catalog.loc[iden, "DEC"]
        zconf = catalog.loc[iden, "zconf"]

        # logM if present
        logM = catalog.loc[iden, "logM"] if "logM" in catalog.columns else np.nan

        # -----------------------------------------------
        # Luminosity Distance
        # -----------------------------------------------
        DL_Mpc, DL_cm = luminosity_distance(z_sys)

        # -----------------------------------------------
        # Load fit
        # -----------------------------------------------
        df = pd.read_csv(fit_path)

        centers = df["Center"].values
        flux = df["Flux"].values
        dflux = df["d_Flux"].values

        exp = {k: v*(1+z_sys) for k,v in REST.items()}
        idx = {k: get_line_fast(centers, exp[k]) for k in REST}

        lines = {}
        for k in REST:
            lines[k] = {
                "center": centers[idx[k]],
                "flux": flux[idx[k]],
                "dflux": dflux[idx[k]]
            }

        # -----------------------------------------------
        # Quality checks (basic S/N)
        # -----------------------------------------------

        SN = {k: lines[k]["flux"]/lines[k]["dflux"] 
              if lines[k]["dflux"] > 0 else 0 for k in REST}

        flag_Hb_SN = SN["Hb"] < 3
        flag_Ha_SN = SN["Ha"] < 3
        flag_N2_SN = SN["N2"] < 3
        flag_O3_SN = SN["O3"] < 3
        flag_SIIa_SN = SN["SIIa"] < 3
        flag_SIIb_SN = SN["SIIb"] < 3

        flag_negative_flux = any(lines[k]["flux"] <= 0 for k in REST)
        
        # -----------------------------------------------
        # Redshifts per line
        # -----------------------------------------------
        z_lines = {
            k: lines[k]["center"]/REST[k] - 1
            for k in REST
        }

        # -----------------------------------------------
        # Rest wavelengths inferred
        # -----------------------------------------------
        rest_obs = {
            k: lines[k]["center"]/(1+z_sys)
            for k in REST
        }


        # -----------------------------------------------
        # Spectrum for EW
        # -----------------------------------------------
        spec_name = f"id{iden}_{xcoord}_{ycoord}_circ_{radius}{smooth}"
        spec_path = os.path.join(SPECTRA_DIR, spec_name)

        spec = np.loadtxt(spec_path)
        wave = spec[:,0]
        flux_spec = spec[:,1]

        Hb_center = lines["Hb"]["center"]

        blue = (wave > Hb_center-90) & (wave < Hb_center-10)
        red  = (wave > Hb_center+10) & (wave < Hb_center+90)

        cont_blue = np.median(flux_spec[blue])
        cont_red = np.median(flux_spec[red])
        cont = 0.5*(cont_blue+cont_red)

        err_blue = np.std(flux_spec[blue])
        err_red = np.std(flux_spec[red])
        cont_err = 0.5*np.sqrt(err_blue**2 + err_red**2)

        EW_obs = lines["Hb"]["flux"]/cont
        sigma_EW_obs = EW_obs*np.sqrt(
            (lines["Hb"]["dflux"]/lines["Hb"]["flux"])**2 +
            (cont_err/cont)**2
        )

        EW_rest = EW_obs/(1+z_sys)
        sigma_EW_rest = sigma_EW_obs/(1+z_sys)

        # -----------------------------------------------
        # Line ratios
        # -----------------------------------------------
        
        FHb = lines["Hb"]["flux"]
        dFHb = lines["Hb"]["dflux"]

        FHb_corr = FHb + 3*FHb/EW_obs
        dFHb_corr = np.sqrt(
            ((1+3/EW_obs)*dFHb)**2 +
            ((-3*FHb)/EW_obs**2 * sigma_EW_obs)**2
        )


        R1 = lines["O3"]["flux"]/FHb_corr
        R2 = lines["N2"]["flux"]/lines["Ha"]["flux"]
        SII_sum = lines["SIIa"]["flux"] + lines["SIIb"]["flux"]
        R3 = SII_sum/lines["Ha"]["flux"]

        R1_err = R1*np.sqrt(
            (lines["O3"]["dflux"]/lines["O3"]["flux"])**2 +
            (dFHb_corr/FHb_corr)**2
        )

        R2_err = R2*np.sqrt(
            (lines["N2"]["dflux"]/lines["N2"]["flux"])**2 +
            (lines["Ha"]["dflux"]/lines["Ha"]["flux"])**2
        )

        SII_sum_err = np.sqrt(
            lines["SIIa"]["dflux"]**2 +
            lines["SIIb"]["dflux"]**2
        )

        R3_err = R3*np.sqrt(
            (SII_sum_err/SII_sum)**2 +
            (lines["Ha"]["dflux"]/lines["Ha"]["flux"])**2
        )

        logR1 = np.log10(R1)
        logR2 = np.log10(R2)
        logR3 = np.log10(R3)

        logR1_err = R1_err/(R1*LN10)
        logR2_err = R2_err/(R2*LN10)
        logR3_err = R3_err/(R3*LN10)

        O3N2 = logR1 - logR2
        O3N2_err = np.sqrt(logR1_err**2 + logR2_err**2)

        OH = 8.73 - 0.32*O3N2
        OH_err = 0.32*O3N2_err

        # -----------------------------------------------
        # SFR
        # -----------------------------------------------
        FHa = lines["Ha"]["flux"]*1e-20
        FHb = lines["Hb"]["flux"]*1e-20
        dFHa = lines["Ha"]["dflux"]*1e-20
        dFHb = lines["Hb"]["dflux"]*1e-20

        FHb_corr = FHb + 3*FHb/EW_obs
        dFHb_corr = np.sqrt(
            ((1+3/EW_obs)*dFHb)**2 +
            ((-3*FHb)/EW_obs**2 * sigma_EW_obs)**2
        )

        Av = 6.547*np.log10(FHa/(2.86*FHb_corr))
        d_Av = 6.547/LN10*np.sqrt(
            (dFHa/FHa)**2 +
            (dFHb_corr/FHb_corr)**2
        )

        FHa_der = FHa*10**(0.31*Av)
        dFHa_der = FHa_der*np.sqrt(
            (dFHa/FHa)**2 +
            (0.31*LN10*d_Av)**2
        )

        LHa = 4*np.pi*FHa_der*DL_cm**2
        dLHa = LHa*(dFHa_der/FHa_der)

        SFR = 7.9e-42*LHa
        dSFR = SFR*(dFHa_der/FHa_der)

        logSFR = np.log10(SFR)
        # ---- Asymmetric log errors ----
        SFR_low = SFR - dSFR
        SFR_up  = SFR + dSFR

        if SFR_low <= 0:
            SFR_low = np.nan

        logSFR_low = np.log10(SFR_low)
        logSFR_up  = np.log10(SFR_up)

        d_logSFR_low = logSFR - logSFR_low
        d_logSFR_up  = logSFR_up - logSFR
        

        # -----------------------------------------------
        # Additional quality flags
        # -----------------------------------------------

        #flag_balmer = (FHa / (2.86 * lines["Hb"]["flux"]*1e-20)) < 0.5
        flag_large_errorR1 = any(x > 5 for x in [logR1_err])
        flag_large_errorR2 = any(x > 5 for x in [logR2_err])
        flag_large_errorR3 = any(x > 5 for x in [logR3_err])
        
        quality_flag = any([
            flag_Hb_SN,
            flag_Ha_SN,
            flag_N2_SN,
            flag_O3_SN,
            flag_negative_flux,
            flag_large_errorR1,
            flag_large_errorR2,
            flag_large_errorR3
        ])

        # -----------------------------------------------
        # SAVE EVERYTHING
        # -----------------------------------------------
        result = pd.DataFrame([{
            "iden": iden,
            "RA": RA,
            "DEC": DEC,
            "z": z_sys,
            "zconf": zconf,
            "logM": logM,
            
            "DL_Mpc": DL_Mpc,
            "DL_cm": DL_cm,
                # --- Emission line measurements ---
            **{
                f"{k}_rest": rest_obs[k]
                for k in rest_obs
            },

            **{
                 f"{k}_center": centers[idx[k]]
                for k in lines
            },

            **{
                f"{k}_flux": flux[idx[k]]
                for k in lines
            },

             **{
                 f"{k}_dflux": dflux[idx[k]]
                for k in lines
            },
            
            "R1": R1,
            "R1_err": R1_err,
            "R2": R2,
            "R2_err": R2_err,
            "R3": R3,
            "R3_err": R3_err,
            
            "logR1": logR1,
            "logR1_err": logR1_err,
            "logR2": logR2,
            "logR2_err": logR2_err,
            "logR3": logR3,
            "logR3_err": logR3_err,
            
            "O3N2": O3N2,
            "O3N2_err": O3N2_err,
            "12+log(O/H)": OH,
            "12+log(O/H)_err": OH_err,
            
            "EW_Hb_obs": EW_obs,
            "EW_Hb_obs_err": sigma_EW_obs,
            "EW_Hb_rest": EW_rest,
            "EW_Hb_rest_err": sigma_EW_rest,
            
            "SFR": SFR,
            "d_SFR": dSFR,
            "log10_SFR": logSFR,
            "d_log10_SFR_up": d_logSFR_up,
            "d_log10_SFR_low": d_logSFR_low,
        
            "flag_Hb_SN": flag_Hb_SN,
            "flag_Ha_SN": flag_Ha_SN,
            "flag_N2_SN": flag_N2_SN,
            "flag_O3_SN": flag_O3_SN,
            "flag_SIIa_SN": flag_SIIa_SN,
            "flag_SIIb_SN": flag_SIIb_SN,
            #"flag_balmer": flag_balmer,
            "flag_negative_flux": flag_negative_flux,
            "flag_large_errorR1": flag_large_errorR1,
            "flag_large_errorR2": flag_large_errorR2,
            "flag_large_errorR3": flag_large_errorR3,
            "quality_flag": quality_flag
        }])    

        output_name = f"A_new_Z_fit_{base}.csv"
        result.to_csv(os.path.join(LINES_DIR, output_name),
                      index=False, float_format="%.3f")

        print(f"✔ {filename}")

    except Exception as e:
        print(f"❌ Error in {filename}: {e}")


# ======================================================
# MAIN
# ======================================================

def main():
    fit_files = glob.glob(os.path.join(LINES_DIR, "fit_*.csv"))
    for f in fit_files:
        process_fit_file(f)

    print("\nPipeline complete.")


if __name__ == "__main__":
    main()


✔ fit_151_r4_197_147.csv
✔ fit_1103_r5_197_285.csv
✔ fit_1081_r5_335_276.csv
✔ fit_202_r8_272_138.csv
❌ Error in fit_212_r4_283_273.csv: SPECTRA/id212_283_273_circ_r4 not found.
✔ fit_202_r9_271_139.csv
✔ fit_553_r3_147_264.csv
✔ fit_202_r4_273_137.csv
✔ fit_1233_r3_72_279_smooth.csv
✔ fit_812_r10_206_308.csv
✔ fit_407_r6_187_141.csv
✔ fit_1483_r5_123_313.csv
✔ fit_590_r6_293_183.csv
✔ fit_1103_r3_198_284_smooth.csv
✔ fit_680_r4_278_220.csv
✔ fit_212_r11_285_274.csv
✔ fit_151_r7_195_149.csv
✔ fit_812_r2_206_308.csv
✔ fit_1081_r2_335_276.csv
✔ fit_212_r7_283_272.csv
✔ fit_285_r2_233_167.csv
✔ fit_553_r6_148_264.csv
✔ fit_285_r6_234_165_smooth.csv
✔ fit_459_r2_152_157.csv
✔ fit_1233_r4_72_279_smooth.csv
✔ fit_1086_r4_263_273.csv
✔ fit_407_r2_188_141.csv
✔ fit_680_r2_278_220.csv
✔ fit_101_r2_211_42_smooth.csv
✔ fit_459_r5_152_157.csv
✔ fit_553_r5_148_264.csv
✔ fit_590_r2_294_183.csv

Pipeline complete.
